In [0]:
%run /Users/arnaldo.jbj@gmail.com/ElectricalGeneratedDistribution-Project/pipelines/02_extracting_from_bronze_to_treatments_into_silver/step00_IMPORTS

In [0]:
%run /Users/arnaldo.jbj@gmail.com/ElectricalGeneratedDistribution-Project/pipelines/02_extracting_from_bronze_to_treatments_into_silver/step00_AUXILIARY_FUNCTIONS

In [0]:
df = (
       spark.table('bronze.tarifas_sul')
                .withColumn(
                             'uf',
                             F.when(F.col('distribuidora') == 'Copel', 'PR')
                              .when(F.col('distribuidora') == 'CELESC-DIS', 'SC')
                           )
                .withColumnsRenamed(
                                     {
                                       'distribuidora': 'distributor',
                                       'vigencia_inicio': 'start_date',
                                       'vigencia_fim' : 'end_date',
                                       'te_rs_mwh' : 'te_real_by_kwh',
                                       'tusd_rs_mwh' : 'tusd_real_by_kwh',
                                       'fio_b_rs_mwh' : 'tusd_wire_b_real_by_kwh',
                                       'aliquota_icms' : 'icms_tax',
                                       'pis_cofins' : 'pis_cofins_tax'
                                     }
                                   )
     )

df.show(truncate=False)

In [0]:
list_regional = ['centro_oeste', 'nordeste', 'sul', 'sudeste']

# Check all bronze tables 
all_bronze_tables = [r.tableName for r in spark.sql("SHOW TABLES IN bronze").collect()]

df_fee = None
for reg in list_regional:
    table_origin = f"tarifas_{reg}"

    # Check if table exists
    if table_origin not in all_bronze_tables:
        print(f"bronze.{table_origin} not found — skipping")
        continue

    # Reading table
    df_bronze = spark.read.table(f"bronze.{table_origin}")

    # Restore canonical column names when accents were corrupted (U+FFFD) at bronze ingestion
    ren_accents = {}
    for name in ['Início_Vigência', 'Fim_Vigência', 'Alíquota_ICMS']:
        corrupted = ''.join('\ufffd' if ord(ch) > 127 else ch for ch in name)
        if corrupted in df_bronze.columns:
            ren_accents[corrupted] = name
    if ren_accents:
        df_bronze = df_bronze.withColumnsRenamed(ren_accents)

    # Detect source table format via "_source_file" field
    source_file = df_bronze.select('_source_file').first()['_source_file']
    is_json = source_file is not None and source_file.endswith('.json')

    if is_json:
        # JSON: explode the 'tarifas' array and flatten nested structs into columns
        df = (
                df_bronze.withColumnsRenamed({'tarifas': 'fees',})
                    .withColumn('regional', F.lit('co'))
                    .withColumn('fees', F.explode('fees'))
                    .select(
                             'regional',
                             F.col('fees.distribuidora.nome').alias('distributor'),
                             F.col('fees.distribuidora.uf').alias('uf'),
                             F.col('fees.componentes.te').alias('te_real_by_kwh'),
                             F.col('fees.componentes.tusd').alias('tusd_real_by_kwh'),
                             F.col('fees.componentes.tusd_fio_b').alias('tusd_wire_b_real_by_kwh'),
                             F.col('fees.tributos.icms').alias('icms_tax'),
                             F.col('fees.tributos.pis_cofins').alias('pis_cofins_tax'),
                             F.col('fees.vigencia.inicio').alias('start_date'),
                             F.col('fees.vigencia.fim').alias('end_date'),
                             '_source_file',
                             '_source_path',
                             '_ingestion_ts'
                           )
             )
    else:
        # CSV: columns are already flat, just add regional
        if reg == 'nordeste':
          df = (
                 df_bronze.withColumn('Início_Vigência', F.date_format(F.to_date(F.col('Início_Vigência'), 'dd/MM/yyyy'), 'yyyy-MM-dd'))
                          .withColumn(
                                      'Fim_Vigência', 
                                      F.date_format(
                                                     F.try_to_date(
                                                                    F.regexp_replace(F.col('Fim_Vigência'), 'vigente', ''), 
                                                                    'dd/MM/yyyy'
                                                                  ), 
                                                     'yyyy-MM-dd'
                                                   )
                                     )
                          .withColumn(
                                       'TE__R$/MWh_',
                                       F.regexp_replace(F.col('TE__R$/MWh_'), ',', '.').cast('double') / F.lit(1000)
                                     )
                          .withColumn(
                                       'TUSD__R$/MWh_',
                                       F.regexp_replace(F.col('TUSD__R$/MWh_'), ',', '.').cast('double') / F.lit(1000)
                                     )
                          .withColumn(
                                       'TUSD_Fio_B__R$/MWh_',
                                       F.regexp_replace(F.col('TUSD_Fio_B__R$/MWh_'), ',', '.').cast('double') / F.lit(1000)
                                     )
                          .withColumn(
                                       'Alíquota_ICMS',
                                       F.regexp_replace(F.regexp_replace(F.col('Alíquota_ICMS'), ',', '.'), '%', '').cast('double') / F.lit(100)
                                     )
                          .withColumn(
                                       'PIS/COFINS',
                                       F.regexp_replace(F.regexp_replace(F.col('PIS/COFINS'), ',', '.'), '%', '').cast('double') / F.lit(100)
                                     )
                          .withColumn('regional' ,F.lit('nordeste'))
                          .withColumnsRenamed(
                                               {
                                                 'Distribuidora': 'distributor',
                                                 'Estado': 'uf',
                                                 'Início_Vigência': 'start_date',
                                                 'Fim_Vigência' : 'end_date',
                                                 'TE__R$/MWh_' : 'te_real_by_kwh',
                                                 'TUSD__R$/MWh_' : 'tusd_real_by_kwh',
                                                 'TUSD_Fio_B__R$/MWh_' : 'tusd_wire_b_real_by_kwh',
                                                 'Alíquota_ICMS' : 'icms_tax',
                                                 'PIS/COFINS' : 'pis_cofins_tax'
                                               }
                                             )
               )
        elif reg == 'sudeste':
          df = (
                 df_bronze.withColumn('INICIO_VIGENCIA', F.date_format(F.to_date(F.col('INICIO_VIGENCIA'), 'dd/MM/yyyy'), 'yyyy-MM-dd'))
                          .withColumn('FIM_VIGENCIA', F.date_format(F.to_date(F.col('FIM_VIGENCIA'), 'dd/MM/yyyy'), 'yyyy-MM-dd'))
                          .withColumn(
                                       'ICMS',
                                       F.regexp_replace(F.regexp_replace(F.col('ICMS'), ',', '.'), '%', '').cast('double') / F.lit(100)
                                     )
                          .withColumn(
                                       'PIS_COFINS',
                                       F.regexp_replace(F.regexp_replace(F.col('PIS_COFINS'), ',', '.'), '%', '').cast('double')
                                     )
                          .withColumn(
                                       'TE_RS_KWH',
                                       F.regexp_replace(F.col('TE_RS_KWH'), ',', '.').cast('double')
                                     )
                          .withColumn(
                                       'TUSD_RS_KWH',
                                       F.regexp_replace(F.col('TUSD_RS_KWH'), ',', '.').cast('double')
                                     )
                          .withColumn(
                                       'TUSD_FIO_B_RS_KWH',
                                       F.regexp_replace(F.col('TUSD_FIO_B_RS_KWH'), ',', '.').cast('double')
                                     )
                          .withColumn('regional' ,F.lit('sudeste'))
                          .withColumnsRenamed(
                                               {
                                                 'DISTRIBUIDORA': 'distributor',
                                                 'UF': 'uf',
                                                 'INICIO_VIGENCIA': 'start_date',
                                                 'FIM_VIGENCIA' : 'end_date',
                                                 'TE_RS_KWH' : 'te_real_by_kwh',
                                                 'TUSD_RS_KWH' : 'tusd_real_by_kwh',
                                                 'TUSD_FIO_B_RS_KWH' : 'tusd_wire_b_real_by_kwh',
                                                 'ICMS' : 'icms_tax',
                                                 'PIS_COFINS' : 'pis_cofins_tax'
                                               }
                                             )
               )
        elif reg == 'sul':
          df = (
                  df_bronze.withColumn(
                                         'uf',
                                         F.when(F.col('distribuidora') == 'Copel', 'PR')
                                          .when(F.col('distribuidora') == 'CELESC-DIS', 'SC')
                                       )
                           .withColumn('regional' ,F.lit('sul'))
                           .withColumn('te_rs_mwh', F.col('te_rs_mwh').cast('double') / F.lit(1000))
                           .withColumn('tusd_rs_mwh', F.col('tusd_rs_mwh').cast('double') / F.lit(1000))
                           .withColumn('fio_b_rs_mwh', F.col('fio_b_rs_mwh').cast('double') / F.lit(1000))
                           .withColumnsRenamed(
                                                {
                                                  'distribuidora': 'distributor',
                                                  'vigencia_inicio': 'start_date',
                                                  'vigencia_fim' : 'end_date',
                                                  'te_rs_mwh' : 'te_real_by_kwh',
                                                  'tusd_rs_mwh' : 'tusd_real_by_kwh',
                                                  'fio_b_rs_mwh' : 'tusd_wire_b_real_by_kwh',
                                                  'aliquota_icms' : 'icms_tax',
                                                  'pis_cofins' : 'pis_cofins_tax'
                                                }
                                              )
                           .drop('te_rs_mwh')
               )


    if df_fee is None:
        df_fee = df
    else:
        df_fee = df_fee.unionByName(df, allowMissingColumns=True)

    print(f"✓ {table_origin} ({'JSON' if is_json else 'CSV'})")

In [0]:
df_fee = check_and_remove_duplicates(df_fee,['uf','start_date','end_date'])

In [0]:
w = Window.partitionBy('uf').orderBy('start_date')

df_fee = (
            df_fee.withColumn('start_date', F.to_date('start_date'))
                  .withColumn('end_date', F.to_date('end_date'))
                  .withColumn('prev_end_date', F.lag('end_date').over(w))
                  .withColumn(
                                 'start_date',
                                 F.when(
                                            (F.col('prev_end_date').isNotNull())
                                          & (F.col('start_date') <= F.col('prev_end_date')),
                                          F.date_add(F.col('prev_end_date'), 1)
                                       ).otherwise(F.col('start_date'))
                             )
                  .drop('prev_end_date')
         )

In [0]:
check_duplicates(df_fee,['uf','start_date','end_date'])

In [0]:
if df_fee is not None:
    table_destiny = "silver.fees"
    df_fee.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_destiny)
    print(f"\n✓ {table_destiny} written ({df_fee.count()} rows)\n")